In [0]:
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://statistics.gov.scot/downloads/cube-table?uri=http://statistics.gov.scot/data/gross-domestic-product-quarterly-output-by-industry"

response = requests.get(url)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))

df_raw = df_raw.rename(columns={"Industry Sector (SIC 07)": "industry_sector"})

print(df_raw.shape)
print(df_raw.dtypes)
df_raw.head(10)

In [0]:
df_bronze = spark.createDataFrame(df_raw)

df_bronze.printSchema()
df_bronze.select("*").show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.scottish_gdp")
)
